# Atari DQN Training
Runs the DQN training loop from the DeepMind 2015 paper on Atari Breakout.

In [9]:
!nvidia-smi

Tue Jun 30 10:39:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   58C    P0             30W /   72W |     280MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
import psutil

ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total/1e9:.1f}GB")
print(f"Available: {ram.available/1e9:.1f}GB")

Total RAM: 56.9GB
Available: 53.7GB


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
!git pull

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 621 bytes | 621.00 KiB/s, done.
From https://github.com/pradsgit/atari-dqn
   83ad814..6eac7d6  main       -> origin/main
Updating 83ad814..6eac7d6
Fast-forward
 env.py | 13 +++++--------
 1 file changed, 5 insertions(+), 8 deletions(-)


In [5]:
import os

if not os.path.exists('/content/atari-dqn'):
    !git clone https://github.com/pradsgit/atari-dqn.git /content/atari-dqn

os.chdir('/content/atari-dqn')
print('Working directory:', os.getcwd())

Working directory: /content/atari-dqn


In [6]:
!pip install -q "gymnasium[atari]" ale-py opencv-python torch torchvision wandb

In [7]:
import wandb

wandb.login()  # paste your API key from wandb.ai/authorize


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: saipradeep-40 (saipradeep) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch: 2.11.0+cu128
CUDA available: True
Device: cuda


In [14]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [7]:
# SAMPLE TRAINING RUN PARAMS

import config

config.GAME            = "Breakout"
config.N_ENVS          = 2
config.MAX_STEPS       = 10_000
config.MIN_REPLAY_SIZE = 50_000
config.EPSILON_DECAY   = 1_000
config.REPLAY_SIZE     = 50_000
# config.CHECKPOINT_DIR  = "/content/drive/MyDrive/atari-dqn/checkpoints"
# config.VIDEO_DIR       = "/content/drive/MyDrive/atari-dqn/videos"


In [7]:
import os
print(os.cpu_count())

12


In [11]:
import config

config.GAME            = "Breakout"
config.N_ENVS          = 12 
config.MAX_STEPS       = 10_000_000
config.MIN_REPLAY_SIZE = 50_000
config.EPSILON_DECAY   = 1_000_000
config.REPLAY_SIZE     = 600_000
config.CHECKPOINT_DIR  = "/content/drive/MyDrive/atari-dqn/checkpoints"
config.VIDEO_DIR       = "/content/drive/MyDrive/atari-dqn/videos"


In [12]:
import tracemalloc
tracemalloc.start()

import train

# run training
train.train()

snapshot = tracemalloc.take_snapshot()
for stat in snapshot.statistics('lineno')[:10]:
    print(stat)

using device: cuda, n_envs: 12


episode    1 | steps     1440 | reward 0.0 | epsilon 0.999 | loss collecting | cpu 2.8GB | gpu 0.0GB
episode    2 | steps     1440 | reward 0.0 | epsilon 0.999 | loss collecting | cpu 2.8GB | gpu 0.0GB
episode    3 | steps     1440 | reward 0.0 | epsilon 0.999 | loss collecting | cpu 2.8GB | gpu 0.0GB
episode    4 | steps     1440 | reward 0.0 | epsilon 0.999 | loss collecting | cpu 2.8GB | gpu 0.0GB
episode    5 | steps     2004 | reward 1.0 | epsilon 0.998 | loss collecting | cpu 2.8GB | gpu 0.0GB
episode    6 | steps     2004 | reward 1.0 | epsilon 0.998 | loss collecting | cpu 2.8GB | gpu 0.0GB
episode    7 | steps     2028 | reward 1.0 | epsilon 0.998 | loss collecting | cpu 2.8GB | gpu 0.0GB
episode    8 | steps     2160 | reward 2.0 | epsilon 0.998 | loss collecting | cpu 2.8GB | gpu 0.0GB
episode    9 | steps     2340 | reward 2.0 | epsilon 0.998 | loss collecting | cpu 2.9GB | gpu 0.0GB
episode   10 | steps     2352 | reward 2.0 | epsilon 0.998 | loss collecting | cpu 2.9GB | 

cpu_ram_gb,▁▂▂▂▂▃▅▅▅▅▇█████████████████████████████
episode,▁▁▁▁▁▁▁▂▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
epsilon,██▇▇▆▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
gpu_ram_gb,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▃▃▄▄▄▄▄▄▅▅▆▆▆▇▇▇▇▇▇██▇█
mean_max_q,▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▃▅▅▅▄▅▆▆▆▅▆▆▇▇▇▆▇█▇▇█▇██▇
reward,▁▁▁▁▁▁▁▁▁▁▂▂▂▃▅▅▅▇▅▄▅▄▅▇█▅█▃▄█▅█▄▅▄▇▇▃▄▇
total_steps,▁▁▁▁▁▁▁▁▁▁▂▂▂▂▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█
cpu_ram_gb,37.4284
episode,9724
epsilon,0.1


/content/atari-dqn/replay_buffer.py:25: size=1346 MiB, count=2, average=673 MiB
/content/atari-dqn/replay_buffer.py:24: size=1346 MiB, count=2, average=673 MiB
<frozen importlib._bootstrap_external>:757: size=77.0 MiB, count=564168, average=143 B
/usr/lib/python3.12/linecache.py:142: size=5498 KiB, count=59754, average=94 B
<frozen importlib._bootstrap>:488: size=4994 KiB, count=51818, average=99 B
/usr/lib/python3.12/dataclasses.py:473: size=2507 KiB, count=25399, average=101 B
/usr/local/lib/python3.12/dist-packages/lark/visitors.py:284: size=2223 KiB, count=48594, average=47 B
/usr/local/lib/python3.12/dist-packages/torch/__init__.py:2167: size=2159 KiB, count=15092, average=146 B
/usr/local/lib/python3.12/dist-packages/lark/visitors.py:180: size=2091 KiB, count=48594, average=44 B
<frozen abc>:106: size=1804 KiB, count=6049, average=305 B


In [31]:
import importlib.util, traceback
spec = importlib.util.spec_from_file_location("dqn_evaluate", "/content/atari-dqn/dqn_evaluate.py")
mod = importlib.util.module_from_spec(spec)
try:
    spec.loader.exec_module(mod)
except Exception:
    traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_3987/2594089552.py", line 5, in <cell line: 0>
    spec.loader.exec_module(mod)
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/content/atari-dqn/dqn_evaluate.py", line 5, in <module>
    from env import AtariEnv
ModuleNotFoundError: No module named 'env'


In [1]:
import os, sys
import numpy as np

if not os.path.exists('/content/atari-dqn'):
    os.system('git clone https://github.com/pradsgit/atari-dqn.git /content/atari-dqn')

os.chdir('/content/atari-dqn')
if '/content/atari-dqn' not in sys.path:
    sys.path.insert(0, '/content/atari-dqn')

import dqn_evaluate as evaluate
import torch

evaluate.CHECKPOINT_DIR = "/content/drive/MyDrive/atari-dqn/checkpoints"
evaluate.VIDEO_DIR      = "/content/drive/MyDrive/atari-dqn/videos"

device = "cuda" if torch.cuda.is_available() else "cpu"

checkpoints = sorted(
    [f for f in os.listdir(evaluate.CHECKPOINT_DIR) if f.endswith(".pt")],
    key=lambda f: int(f.split("_")[-1].replace(".pt", ""))
)
print(f"found {len(checkpoints)} checkpoints:")
for c in checkpoints:
    print(" ", c)

latest = os.path.join(evaluate.CHECKPOINT_DIR, checkpoints[-1])
agent = evaluate.load_agent(latest, device)


env = __import__('env').AtariEnv("Breakout", clip_rewards=False)
state = env.reset()

# check q-values and action distribution over 200 steps
actions = []
for _ in range(200):
    with torch.no_grad():
        q = agent.online_net(torch.tensor(state).unsqueeze(0).to(device))
    action = torch.argmax(q).item()
    actions.append(action)
    state, _, done, _ = env.step(action)
    if done:
        state = env.reset()

from collections import Counter
print("action distribution:", Counter(actions))
print("q-values for last state:", q.cpu().numpy())


mean, std = evaluate.evaluate(agent, n_episodes=3)
evaluate.record_video(agent, n_episodes=1)

found 34 checkpoints:
  dqn_step_10000.pt
  dqn_step_100008.pt
  dqn_step_200016.pt
  dqn_step_300024.pt
  dqn_step_400032.pt
  dqn_step_500040.pt
  dqn_step_600048.pt
  dqn_step_700056.pt
  dqn_step_800064.pt
  dqn_step_900072.pt
  dqn_step_1000080.pt
  dqn_step_1100088.pt
  dqn_step_1200096.pt
  dqn_step_1300104.pt
  dqn_step_1400112.pt
  dqn_step_1500120.pt
  dqn_step_1600128.pt
  dqn_step_1700136.pt
  dqn_step_1800144.pt
  dqn_step_1900152.pt
  dqn_step_2000160.pt
  dqn_step_2100168.pt
  dqn_step_2200176.pt
  dqn_step_2300184.pt
  dqn_step_2400192.pt
  dqn_step_2500200.pt
  dqn_step_2600208.pt
  dqn_step_2700216.pt
  dqn_step_2800224.pt
  dqn_step_2900232.pt
  dqn_step_3000240.pt
  dqn_step_3100248.pt
  dqn_step_3200256.pt
  dqn_step_3300264.pt
loaded checkpoint from step 3300264, episode 13684
action distribution: Counter({2: 92, 1: 58, 3: 31, 0: 19})
q-values for last state: [[1.269748  1.2737514 1.2844017 1.2807282]]
eval episode   1 | reward 5.0
eval episode   2 | reward 5.0
ev

In [11]:
import dqn_evaluate as evaluate

evaluate.CHECKPOINT_DIR = "/content/drive/MyDrive/atari-dqn/checkpoints"
evaluate.VIDEO_DIR      = "/content/drive/MyDrive/atari-dqn/videos"

# patch select_action to use epsilon=0.3 so paddle actually moves
import types

def select_action_with_noise(self, state, epsilon):
    import random, torch
    if random.random() < 0.3:
        return random.randint(0, self.n_actions - 1)
    with torch.no_grad():
        output = self.online_net(torch.tensor(state).unsqueeze(0).to(self.device))
    return torch.argmax(output).item()

agent.select_action = types.MethodType(select_action_with_noise, agent)
evaluate.record_video(agent, n_episodes=1)

recorded episode 1 | reward 2.0
videos saved to /content/drive/MyDrive/atari-dqn/videos
